# concateFile.ipynb

Notebook for advanced ECG concatenation with sequential HR ordering, pure/mixed level segments, and duration-aware truncation.

## Requirements implemented
- Sequential HR ordering (ascending HR number)
- Pure level segments (single label)
- Mixed level segments with majority-vote labels (ties -> lowest label)
- Duration-aware concatenation with smart truncation
- Time continuity preservation and column cleanup

In [4]:
import os
import random
from collections import Counter
from typing import Dict, List, Optional, Tuple

import pandas as pd

In [ ]:
class ECGAdvancedConcatenator:
    """
    Advanced ECG data concatenator with sequential HR ordering and duration-aware truncation.
    """

    def __init__(self, csv_label_file: Optional[str], data_dir: str, labels: Optional[List[int]] = None) -> None:
        if csv_label_file is not None and not os.path.isfile(csv_label_file):
            raise FileNotFoundError(f"CSV label file not found: {csv_label_file}")
        if not os.path.isdir(data_dir):
            raise FileNotFoundError(f"Data directory not found: {data_dir}")

        self.csv_label_file = csv_label_file
        self.data_dir = data_dir
        self.labels = labels or [0, 1, 2, 3]
        self.label_files: Dict[int, List[str]] = {}
        self.data_cache: Dict[str, pd.DataFrame] = {}

        if self.csv_label_file:
            self._load_label_mapping()
        else:
            self._scan_label_directories()

    def _scan_label_directories(self) -> None:
        for label in self.labels:
            label_dir = os.path.join(self.data_dir, str(label))
            if not os.path.isdir(label_dir):
                raise FileNotFoundError(f"Label directory not found: {label_dir}")
            files = [f for f in os.listdir(label_dir) if f.lower().endswith(".csv")]
            if not files:
                raise FileNotFoundError(f"No CSV files found in {label_dir}")
            self.label_files[label] = sorted(files, key=self._extract_hr_number)

    def _load_label_mapping(self) -> None:
        df = pd.read_csv(self.csv_label_file)
        df.columns = [c.strip() for c in df.columns]
        if "File" not in df.columns or "Label" not in df.columns:
            raise ValueError("CSV must include 'File' and 'Label' columns.")

        for _, row in df.iterrows():
            label = int(row["Label"])
            filename = str(row["File"])
            self.label_files.setdefault(label, []).append(filename)

        for label in self.label_files:
            self.label_files[label] = sorted(self.label_files[label], key=self._extract_hr_number)

    def _get_full_path(self, label: int, filename: str) -> str:
        return os.path.join(self.data_dir, str(label), filename)

    def _load_label_files(self, label: int) -> None:
        if label not in self.label_files:
            raise ValueError(f"Label {label} is not available.")

        for filename in self.label_files[label]:
            full_path = self._get_full_path(label, filename)
            if full_path in self.data_cache:
                continue
            if not os.path.isfile(full_path):
                raise FileNotFoundError(f"File missing for label {label}: {full_path}")
            df = pd.read_csv(full_path)
            df.columns = [c.strip() for c in df.columns]
            self.data_cache[full_path] = df

    @staticmethod
    def _get_duration_from_dataframe(df: pd.DataFrame) -> float:
        if "Time" in df.columns:
            time_series = df["Time"].to_numpy()
            if len(time_series) == 0:
                return 0.0
            return float(time_series[-1] - time_series[0])
        return float(len(df))

    @staticmethod
    def _extract_hr_number(filename: str) -> int:
        base = os.path.splitext(os.path.basename(filename))[0]
        digits = "".join([ch for ch in base if ch.isdigit()])
        if not digits:
            raise ValueError(f"Unable to extract HR number from filename: {filename}")
        return int(digits)

    @staticmethod
    def _offset_time(df: pd.DataFrame, offset: float) -> pd.DataFrame:
        if "Time" not in df.columns:
            return df
        df = df.copy()
        df["Time"] = df["Time"] + offset
        return df

    def concatenate_preserve_time(self, label: int, duration_minutes: float, random_order: bool = True) -> pd.DataFrame:
        self._load_label_files(label)
        label_files = list(self.label_files[label])
        if random_order:
            random.shuffle(label_files)
        else:
            label_files = sorted(label_files, key=self._extract_hr_number)

        target_seconds = duration_minutes * 60.0
        total_duration = 0.0
        output_parts = []
        time_offset = 0.0

        for filename in label_files:
            full_path = self._get_full_path(label, filename)
            df = self.data_cache[full_path]
            duration = self._get_duration_from_dataframe(df)

            remaining = target_seconds - total_duration
            if remaining <= 0:
                break

            if duration <= remaining:
                output_parts.append(self._offset_time(df, time_offset))
                total_duration += duration
                if "Time" in df.columns and len(df) > 0:
                    time_offset = output_parts[-1]["Time"].iloc[-1]
                continue

            # Truncate last file to match the remaining duration.
            truncated = df.copy()
            if "Time" in truncated.columns:
                start_time = truncated["Time"].iloc[0]
                cutoff = start_time + remaining
                truncated = truncated[truncated["Time"] <= cutoff]
                if len(truncated) > 0:
                    truncated["Time"] = truncated["Time"] - truncated["Time"].iloc[0]
            else:
                truncated = truncated.iloc[: int(remaining)]
            output_parts.append(self._offset_time(truncated, time_offset))
            total_duration = target_seconds
            break

        if not output_parts:
            raise ValueError(f"No data available for label {label}.")

        return pd.concat(output_parts, ignore_index=True)

    def concatenate_sequential_hr(self, labels: List[int], num_segments: int, output_dir: str, target_minutes: float) -> None:
        if not labels:
            raise ValueError("Labels list cannot be empty.")
        if num_segments <= 0:
            raise ValueError("num_segments must be positive.")

        os.makedirs(output_dir, exist_ok=True)

        labeled_files: List[Tuple[int, str]] = []
        for label in labels:
            self._load_label_files(label)
            for filename in self.label_files[label]:
                labeled_files.append((label, filename))

        labeled_files.sort(key=lambda item: (self._extract_hr_number(item[1]), item[0]))

        target_seconds = target_minutes * 60.0
        segments: List[Tuple[int, str]] = []
        concat_index = 1

        for label, filename in labeled_files:
            segments.append((label, filename))
            if len(segments) < num_segments:
                continue

            output_df, majority_label = self._build_duration_segment(segments, target_seconds)
            if len(labels) > 1:
                final_dir = os.path.join(output_dir, f"mixed_{majority_label}")
                file_name = f"mixed_{majority_label}_{concat_index:03d}.csv"
            else:
                final_dir = output_dir
                file_name = f"concat_{majority_label}_{concat_index:03d}.csv"
            os.makedirs(final_dir, exist_ok=True)

            output_path = os.path.join(final_dir, file_name)
            output_df.to_csv(output_path, index=False)

            concat_index += 1
            segments = []

    def _build_duration_segment(self, segments: List[Tuple[int, str]], target_seconds: float) -> Tuple[pd.DataFrame, int]:
        label_counts = Counter([label for label, _ in segments])
        majority_label = sorted(label_counts.items(), key=lambda item: (-item[1], item[0]))[0][0]

        output_parts = []
        total_duration = 0.0
        time_offset = 0.0

        for label, filename in segments:
            full_path = self._get_full_path(label, filename)
            df = self.data_cache[full_path]
            duration = self._get_duration_from_dataframe(df)
            remaining = target_seconds - total_duration
            if remaining <= 0:
                break
            if duration <= remaining:
                output_parts.append(self._offset_time(df, time_offset))
                total_duration += duration
                if "Time" in df.columns and len(df) > 0:
                    time_offset = output_parts[-1]["Time"].iloc[-1]
                continue

            truncated = df.copy()
            if "Time" in truncated.columns:
                start_time = truncated["Time"].iloc[0]
                cutoff = start_time + remaining
                truncated = truncated[truncated["Time"] <= cutoff]
                if len(truncated) > 0:
                    truncated["Time"] = truncated["Time"] - truncated["Time"].iloc[0]
            else:
                truncated = truncated.iloc[: int(remaining)]
            output_parts.append(self._offset_time(truncated, time_offset))
            total_duration = target_seconds
            break

        if not output_parts:
            raise ValueError("Unable to build concatenated segment from provided files.")

        return pd.concat(output_parts, ignore_index=True), majority_label

## Usage examples

In [7]:
concatenator = ECGAdvancedConcatenator(
    csv_label_file=None,
    data_dir='data/raw_gen/'
)

# Pure level concatenation
concatenator.concatenate_sequential_hr([0], 5, 'data/concatenated/pure_0/', target_minutes=5)
concatenator.concatenate_sequential_hr([1], 5, 'data/concatenated/pure_1/', target_minutes=5)
concatenator.concatenate_sequential_hr([2], 5, 'data/concatenated/pure_2/', target_minutes=5)
concatenator.concatenate_sequential_hr([3], 5, 'data/concatenated/pure_3/', target_minutes=5)

# Mixed level concatenation (grouped by majority label)
concatenator.concatenate_sequential_hr([0, 1], 5, 'data/concatenated/', target_minutes=5)
concatenator.concatenate_sequential_hr([0, 2], 5, 'data/concatenated/', target_minutes=5)
concatenator.concatenate_sequential_hr([1, 2], 5, 'data/concatenated/', target_minutes=5)
concatenator.concatenate_sequential_hr([0, 1, 2], 5, 'data/concatenated/', target_minutes=5)
concatenator.concatenate_sequential_hr([1, 2, 3], 5, 'data/concatenated/', target_minutes=5)
concatenator.concatenate_sequential_hr([0, 1, 2, 3], 5, 'data/concatenated/', target_minutes=5)